# Unsloth Minimal Training Smoke Test

This notebook shows the smallest useful training loop behind an **Unsloth-style** workflow: a frozen-or-small language model plus tiny **LoRA** adapter matrices trained on instruction-response text.

The important MacBook note is practical: current official Unsloth docs describe Mac support for Studio chat/data workflows, while Apple MLX training is still listed as coming soon. The same requirements page says Unsloth Core expects support from packages such as `xformers`, `torch`, `BitsAndBytes`, and `triton`, with Apple/Silicon/MLX still in progress. See the [Unsloth requirements](https://unsloth.ai/docs/get-started/fine-tuning-for-beginners/unsloth-requirements) for the current status.

So this notebook checks whether `unsloth` is available, then runs a Mac-safe PyTorch LoRA smoke test that fits comfortably on a MacBook Pro M1 with 16 GB.


## 1. Setup

All hyperparameters live in one place. The defaults deliberately keep the model tiny, the sequence length short, and the training loop brief.


In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 42,  # Random seed for reproducible demo behavior

    # Runtime
    'prefer_mps': True,  # Use Apple Silicon GPU when PyTorch MPS is available

    # Data
    'max_new_tokens': 36,  # Maximum characters to generate after the response prefix

    # Tiny model
    'd_model': 32,  # Embedding width
    'n_heads': 2,  # Attention heads
    'n_layers': 1,  # Transformer blocks
    'lora_rank': 4,  # Low-rank adapter rank
    'lora_alpha': 8,  # LoRA scaling factor
    'dropout': 0.0,  # Keep deterministic for a smoke test

    # Training
    'learning_rate': 3e-3,  # AdamW learning rate
    'max_steps': 100,  # Short run; should finish quickly on M1
    'grad_clip': 1.0,  # Gradient clipping for stability
    'log_every': 20,  # Print loss every N steps
}


### Check the Runtime and Unsloth Availability

This cell is intentionally non-fatal. On an M1 Mac, `unsloth` training may not be importable or usable for Core training, so the notebook continues with the tiny local training loop.


In [ ]:
import importlib.util
import platform
import sys

import torch

print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'PyTorch: {torch.__version__}')
print(f'MPS built: {torch.backends.mps.is_built()}')
print(f'MPS available: {torch.backends.mps.is_available()}')

unsloth_spec = importlib.util.find_spec('unsloth')
print(f'Unsloth package importable: {unsloth_spec is not None}')

if unsloth_spec is not None:
    try:
        from unsloth import FastLanguageModel  # noqa: F401

        print('FastLanguageModel import: ok')
    except Exception as exc:
        print(f'FastLanguageModel import failed: {type(exc).__name__}: {exc}')
else:
    print('Continuing with the Mac-safe LoRA smoke test below.')


### Seed and Device

The model is small enough for CPU, but MPS usually runs this quickly on Apple Silicon. If MPS gives trouble on a local machine, set `CONFIG['prefer_mps'] = False` and rerun.


In [ ]:
import random

import numpy as np

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])

if CONFIG['prefer_mps'] and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')


## 2. Tiny Instruction Data

A real Unsloth fine-tune would use a pretrained model and a larger formatted dataset. For a smoke test, we only need enough examples to prove that loss decreases and generation changes.


In [ ]:
TRAIN_EXAMPLES = [
    ('Classify sentiment: I loved the fast setup.', 'positive'),
    ('Classify sentiment: The answer was late and vague.', 'negative'),
    ('Classify sentiment: It shipped on Tuesday.', 'neutral'),
    ('Classify topic: Reset my password please.', 'account'),
    ('Classify topic: I need a refund for an invoice.', 'billing'),
    ('Classify topic: Where is my package?', 'shipping'),
    ('Rewrite politely: send the logs now.', 'Please send the logs when you can.'),
    ('Rewrite briefly: the build failed because tests failed.', 'Tests failed, so the build failed.'),
]

def format_example(instruction: str, response: str) -> str:
    return f'### Instruction:\n{instruction}\n### Response:\n{response}\n'

texts = [format_example(instruction, response) for instruction, response in TRAIN_EXAMPLES]
print(texts[0])
print(f'Examples: {len(texts)}')


### Character Tokenizer

A character tokenizer is enough for a no-download smoke test. It keeps the notebook self-contained and avoids external model files.


In [ ]:
SPECIAL_TOKENS = ['<pad>', '<bos>', '<eos>']
chars = sorted(set(''.join(texts)))
vocab = SPECIAL_TOKENS + chars
stoi = {token: index for index, token in enumerate(vocab)}
itos = {index: token for token, index in stoi.items()}

pad_id = stoi['<pad>']
bos_id = stoi['<bos>']
eos_id = stoi['<eos>']
space_id = stoi[' ']

def encode(text: str, add_eos: bool = True) -> list[int]:
    ids = [bos_id]
    ids.extend(stoi.get(char, space_id) for char in text)
    if add_eos:
        ids.append(eos_id)
    return ids

def decode(ids: list[int]) -> str:
    pieces = []
    for index in ids:
        if index in (pad_id, bos_id, eos_id):
            continue
        pieces.append(itos[index])
    return ''.join(pieces)

encoded = [encode(text) for text in texts]
max_len = max(len(ids) for ids in encoded)
print(f'Vocab size: {len(vocab)}')
print(f'Max sequence length: {max_len}')


### Build One Full-Batch Dataset

The whole dataset fits in one batch. Targets are shifted by one token, and padding targets use `-100` so PyTorch ignores them in cross-entropy.


In [ ]:
input_rows = []
target_rows = []

for ids in encoded:
    padding = [pad_id] * (max_len - len(ids))
    input_rows.append(ids[:-1] + padding)
    target_rows.append(ids[1:] + [-100] * len(padding))

input_ids = torch.tensor(input_rows, dtype=torch.long, device=device)
target_ids = torch.tensor(target_rows, dtype=torch.long, device=device)

assert input_ids.shape == target_ids.shape
assert input_ids.ndim == 2
print(f'Input shape: {tuple(input_ids.shape)}')
print(f'Target shape: {tuple(target_ids.shape)}')
print(decode(input_ids[0].detach().cpu().tolist()))


## 3. LoRA Building Block

**LoRA** adds a low-rank update to a regular linear layer. Unsloth speeds up this kind of adapter fine-tuning with optimized kernels; here we implement the tiny version directly so the notebook works on a Mac.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class LoRALinear(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = CONFIG['lora_rank'],
        alpha: int = CONFIG['lora_alpha'],
        bias: bool = False,
    ):
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=bias)
        self.lora_a = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_b = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_output = self.base(x)
        adapter_hidden = F.linear(x, self.lora_a)
        adapter_output = F.linear(adapter_hidden, self.lora_b) * self.scaling
        return base_output + adapter_output


### Tiny Causal Language Model

This is a one-block GPT-style model. The query and value projections use LoRA layers because those are common adapter targets in LLM fine-tuning.


In [ ]:
import math

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, lora_rank: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = LoRALinear(d_model, d_model, rank=lora_rank, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = LoRALinear(d_model, d_model, rank=lora_rank, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, d_model = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        attention_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        attention_scores = attention_scores.masked_fill(causal_mask, float('-inf'))
        attention_weights = attention_scores.softmax(dim=-1)
        y = attention_weights @ v
        y = y.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        return self.out_proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, lora_rank: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, lora_rank)
        self.ln_2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.ff(self.ln_2(x))
        return x

class TinyLoRALanguageModel(nn.Module):
    def __init__(self, vocab_size: int, context_len: int):
        super().__init__()
        d_model = CONFIG['d_model']
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, CONFIG['n_heads'], CONFIG['lora_rank'])
            for _ in range(CONFIG['n_layers'])
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        _, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_f(x))


### Count Parameters

The model is intentionally tiny. Even in float32, the parameters are far below one megabyte.


In [ ]:
model = TinyLoRALanguageModel(vocab_size=len(vocab), context_len=max_len - 1).to(device)

total_params = sum(param.numel() for param in model.parameters())
lora_params = sum(
    param.numel()
    for name, param in model.named_parameters()
    if 'lora_' in name
)
parameter_mb = total_params * 4 / 1024**2

print(f'Total parameters: {total_params:,}')
print(f'LoRA parameters: {lora_params:,}')
print(f'Approx parameter memory at fp32: {parameter_mb:.3f} MB')


## 4. Minimal Training Loop

This is the whole point of the smoke test: forward pass, cross-entropy loss, backward pass, gradient clipping, optimizer step. If the loss drops, the training stack is working.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
loss_history = []

model.train()
for step in range(1, CONFIG['max_steps'] + 1):
    logits = model(input_ids)
    loss = F.cross_entropy(
        logits.reshape(-1, len(vocab)),
        target_ids.reshape(-1),
        ignore_index=-100,
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
    optimizer.step()

    loss_value = float(loss.detach().cpu())
    loss_history.append(loss_value)
    if step == 1 or step % CONFIG['log_every'] == 0:
        print(f'step {step:03d} | loss {loss_value:.4f}')

print(f'Initial loss: {loss_history[0]:.4f}')
print(f'Final loss: {loss_history[-1]:.4f}')
assert loss_history[-1] < loss_history[0], 'Smoke test expected training loss to decrease.'


### Plot the Loss Curve

A steep drop is expected because the dataset is tiny. This is a runtime check, not a generalization benchmark.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(loss_history, color='#2563eb', linewidth=2)
ax.set_title('Tiny LoRA smoke-test training loss')
ax.set_xlabel('Step')
ax.set_ylabel('Cross-entropy loss')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 5. Generate a Response

Greedy decoding is enough for this demo. We prompt the trained toy model with examples it saw during the smoke test, then stop at `<eos>` or after a short character budget.


In [ ]:
@torch.no_grad()
def generate_response(instruction: str, max_new_tokens: int = CONFIG['max_new_tokens']) -> str:
    model.eval()
    prefix = f'### Instruction:\n{instruction}\n### Response:\n'
    ids = encode(prefix, add_eos=False)

    for _ in range(max_new_tokens):
        context = ids[-(max_len - 1):]
        context_tensor = torch.tensor([context], dtype=torch.long, device=device)
        next_token = int(model(context_tensor)[0, -1].argmax(dim=-1).detach().cpu())
        if next_token == eos_id:
            break
        ids.append(next_token)

    return decode(ids)

for instruction, expected in TRAIN_EXAMPLES[:3]:
    print('-' * 60)
    print(generate_response(instruction))
    print(f'Expected response: {expected}')


## 6. How This Maps to Real Unsloth

In a real CUDA-capable Unsloth notebook, the tiny local model above is replaced by `FastLanguageModel.from_pretrained(...)`, and the handwritten loop is usually replaced by a trainer such as `SFTTrainer`. The conceptual pieces stay the same:

- Format instruction-response examples.
- Tokenize into next-token prediction batches.
- Attach LoRA adapters to attention/MLP projections.
- Run a short SFT loop and verify loss/generation.

For an M1 MacBook Pro with 16 GB, this notebook keeps the smoke test local and reliable while still exercising the core adapter-training path.
